In [1]:
import random
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt

from keras.layers import TextVectorization,Embedding,SimpleRNN,Dense,Dropout
from keras.models import Sequential
from keras.optimizers import Adam

from sklearn.model_selection import train_test_split

2023-12-28 15:11:26.141534: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

In [3]:
df = pd.read_csv('imdb.csv')

In [4]:
test_size = .15
epochs = 50 
batch_size = 1000
vocabulary_size = 10000
sentence_size = 500

In [5]:
def standardize(text):
    result = tf.strings.lower(text)
    result = tf.strings.regex_replace(result,r'<[^>]*>',r' ')
    result = tf.strings.regex_replace(result,r'[^\x20-\x7E]',r'')
    result = tf.strings.regex_replace(result,r'\s+',r' ')
    return result

In [6]:
x_temp, x_test, y_temp, y_test = train_test_split(df['Review'],df['Sentiment'],test_size=test_size,random_state=seed,stratify=df['Sentiment'])
x_train, x_val, y_train, y_val = train_test_split(x_temp,y_temp,test_size=test_size,random_state=seed,stratify=y_temp)

In [7]:
vectorizer = TextVectorization(max_tokens=vocabulary_size,output_sequence_length=sentence_size,standardize=standardize)
vectorizer.adapt(x_train.values)

In [8]:
model = Sequential([
    vectorizer,
    Embedding(input_dim=vocabulary_size,output_dim=8,input_length=sentence_size,mask_zero=True),
    SimpleRNN(16, return_sequences=True),
    Dropout(.25),
    SimpleRNN(8),
    Dropout(.25), 
    Dense(4, activation='linear'),
    Dense(1, activation='sigmoid')
])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 text_vectorization (TextVe  (None, 500)               0         
 ctorization)                                                    
                                                                 
 embedding (Embedding)       (None, 500, 8)            80000     
                                                                 
 simple_rnn (SimpleRNN)      (None, 500, 16)           400       
                                                                 
 simple_rnn_1 (SimpleRNN)    (None, 8)                 200       
                                                                 
 dense (Dense)               (None, 4)                 36        
                                                                 
 dense_1 (Dense)             (None, 1)                 5         
                                                        

In [9]:
optimizer = Adam(learning_rate=.0001)
model.compile(optimizer=optimizer,loss='binary_crossentropy',metrics=['accuracy'])
history = model.fit(x_train,y_train,epochs=epochs,batch_size=batch_size,validation_data=(x_val,y_val)).history

Epoch 1/50
37/37 [==============================] - 21s 510ms/step - loss: 0.7362 - accuracy: 0.5037 - val_loss: 0.7159 - val_accuracy: 0.5101
Epoch 2/50
11/37 [=======>......................] - ETA: 12s - loss: 0.7138 - accuracy: 0.5090

In [ ]:
model.evaluate(x_test,y_test)

In [ ]:
plt.plot(history['loss'])
plt.plot(history['val_loss'])
plt.show()

In [ ]:
plt.plot(history['accuracy'])
plt.plot(history['val_accuracy'])
plt.show()